# SmolVLM Full Fine-tuning

In [ ]:
!pip install -q transformers trl==0.12.1 datasets bitsandbytes peft accelerate
!pip install num2words
# transformers==4.46.3, trl==0.12.1, datasets==3.1.0, bitsandbytes==0.45.0, peft==0.13.2, accelerate==1.1.1
#!pip install -q flash-attn --no-build-isolation

In [ ]:
import glob
import pandas as pd
import os
import torch
import gc
import time
import matplotlib.pyplot as plt
import torch.multiprocessing as mp
from transformers import AutoModelForImageTextToText, AutoProcessor
from datasets import Dataset
from PIL import Image
from trl import SFTConfig, SFTTrainer

mp.set_start_method('spawn')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## 1. Load Dataset

In [ ]:
# Legacy Colab Drive setup removed; use repository-relative paths.

In [ ]:
# Archive extraction removed; clone the repository instead.

In [ ]:
# load annotation text files
train_text_files = glob.glob('data/legacy/qwen2_5_vl_3b_annots/train_annots/*.txt')
test_text_files = glob.glob('data/legacy/qwen2_5_vl_3b_annots/test_annots/*.txt')

train_text_files.sort()
test_text_files.sort()

print(len(train_text_files))
print(len(test_text_files))

### 1.1. Make a dataframe of all images and texts

In [ ]:
def prepare_img_txt_list(text_files, img_folder_path, split='train'):
    all_images, all_texts = [], []

    for text_file in text_files:
        text = open(text_file).read()

        if len(text) > 200:
            text_file_name = text_file.split(os.path.sep)[-1].split('.txt')[0]

            image_file_name = os.path.join(f'{img_folder_path}/{split}/img/', text_file_name+'.jpg')

            all_images.append(image_file_name)
            all_texts.append(text)

    return pd.DataFrame({'image_paths': all_images, 'texts': all_texts})

## making a dataframe of image_path and corresponding text
NUM_TRAIN = 10
NUM_TEST = 5
IMG_FOLDER_PATH = 'data/raw/sroie/SROIE2019'

train_df = prepare_img_txt_list(train_text_files[:NUM_TRAIN], img_folder_path=IMG_FOLDER_PATH,split='train')
test_df = prepare_img_txt_list(test_text_files[:NUM_TEST],img_folder_path=IMG_FOLDER_PATH,split='test')

# example
print(train_df.head(3))

### 1.2. Make dataset
- The `train_dataset['data']` object is a list of lists, where each inner list represents a single data sample for training.
- Each inner list contains dictionaries representing messages in a conversation, with the following structure:
```python
train_dataset[ #Outer List Represents the entire dataset.
    [ #Each inner list is a single training example.
        {
            'role': 'system/user/assistant',
            'content': [
                {
                    'type': 'text',
                    'text': system_message,user_message etc.,
                    'image': None
                }
            ],
        }
    ]
]
```

In [ ]:
def format_data(samples,system_message,user_message):
    data_samples = []

    for i in range(len(samples['image_paths'])):
        image = samples['image_paths'][i]
        label = samples['texts'][i]

        data_samples.append([
            {
                'role': 'system',
                'content': [
                    {
                        'type': 'text',
                        'text': system_message
                    }
                ],
            },
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'image': image,
                    },
                    {
                        'type': 'text',
                        'text': user_message,
                    }
                ],
            },
            {
                'role': 'assistant',
                'content': [
                    {
                        'type': 'text',
                        'text': label
                    }
                ],
            },
        ])
    return {'data': data_samples}

# hyperparameters
system_message = 'You are a vision language model expert at creating at OCR of receipts, invoices, and forms.'
user_message = 'OCR this image accurately.'

# Convert pandas DataFrames to Hugging Face Datasets
train_hf_dataset = Dataset.from_pandas(train_df)
test_hf_dataset = Dataset.from_pandas(test_df)

train_dataset = train_hf_dataset.map(
    format_data,
    fn_kwargs={'system_message': system_message, 'user_message': user_message},
    batched=True,
    batch_size=128,
    num_proc=8,
    remove_columns=train_hf_dataset.column_names
)
test_dataset = test_hf_dataset.map(
    format_data,
    fn_kwargs={'system_message': system_message, 'user_message': user_message},
    batched=True,
    batch_size=128,
    num_proc=8,
    remove_columns=test_hf_dataset.column_names
)

In [ ]:
# Example
print(train_dataset['data'][2][0]) #system role
print(train_dataset['data'][2][1]) #user role
print(train_dataset['data'][2][2]) #assistant role

### 1.3. Clear GPU memory before training

In [ ]:
def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f'GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')
    print(f'GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB')

clear_memory()

## 2. Full Fine-Tune the Model using TRL

### 2.1. Load model

In [ ]:
# Load model and tokenizer
model_id = 'HuggingFaceTB/SmolVLM2-256M-Video-Instruct'
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map='cuda',
    dtype=torch.bfloat16,
    _attn_implementation='eager', # Use `flash_attention_2` on Ampere GPUs and above and `eager` on older GPUs.
)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print('trainable params:',trainable_params)
    print('all params:', all_param)
    print('trainable%:',100 * trainable_params / all_param)

# use
print_trainable_parameters(model)

### 2.2. Hyperparameters for SFT config

In [ ]:
# Configure training arguments using SFTConfig
training_args = SFTConfig(
    output_dir='trained_models/full_ft/smolvlm2_256m_fullft_qwen2_5_vl_3b',
    logging_dir='trained_models/full_ft/smolvlm2_256m_fullft_qwen2_5_vl_3b',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,  # Decrease if batch size is increased
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=100,
    eval_steps=100,
    save_steps=100,
    logging_strategy='steps',
    eval_strategy='steps',
    save_strategy='steps',
    save_total_limit=1,
    optim='adamw_torch_fused',
    bf16=True,
    report_to='tensorboard',
    remove_unused_columns=False,
    gradient_checkpointing=True,
    # dataloader_num_workers=4,
    dataset_text_field='',
    dataset_kwargs={'skip_prepare_dataset': True},
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
)

In [ ]:
def collate_fn(examples, device = 'cuda'):
  # Get the image token ID from the processor's tokenizer
  image_token_id = processor.tokenizer.additional_special_tokens_ids[
          processor.tokenizer.additional_special_tokens.index('<image>')]

  # Apply the chat template to each example to format the conversation
  # tokenize=False means it returns a string, not token IDs yet
  texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]

  image_inputs = []
  for example in examples:
    image = Image.open(example[1]['content'][0]['image']).convert('RGB')
    image_inputs.append([image])

  # Process the text and image inputs into a batch format
  batch = processor(
      text=texts,
      images=image_inputs,
      return_tensors='pt',
      padding=True # Pad the sequences to the maximum length in the batch
  ).to(device, dtype=torch.bfloat16)

  # Create labels by cloning the input IDs
  labels = batch['input_ids'].clone()
  # Mask padding tokens in labels so they are ignored during loss calculation
  labels[labels == processor.tokenizer.pad_token_id] = -100
  # Mask image token IDs in labels so they are ignored during loss calculation
  labels[labels == image_token_id] = -100

  # Add the prepared labels to the batch dictionary
  batch['labels'] = labels

  return batch

### 2.3. Load trainer and train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset['data'],
    eval_dataset=test_dataset['data'],
    data_collator=collate_fn,
    tokenizer=processor.tokenizer
)

# train model
trainer.can_return_loss = True
trainer.train()

# save model to local
trainer.save_model(training_args.output_dir)
processor.save_pretrained(training_args.output_dir)
# save model to Hugging face
# trainer.push_to_hub()
# processor.push_to_hub('smolvlm256m_fullft_qwen2_5_vl_3b')

## 3. Testing the Fine-Tuned Model

In [ ]:
clear_memory()

In [ ]:
model_path = 'trained_models/full_ft/smolvlm2_256m_fullft_qwen2_5_vl_3b'
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    device_map='cuda',
    dtype=torch.bfloat16,
    _attn_implementation='eager', # Use `flash_attention_2` on Ampere GPUs and above and `eager` on older GPUs.
)

processor = AutoProcessor.from_pretrained(model_path)

### 3.1. Generate text from Image and Message
1. Pass image as RGB image
2. Message:
    - Using `apply_chat_template`
    - The input to apply_chat_template should be structured as a list of dictionaries with `role` and `content` keys.
    - The common roles are:
        - `system` for directives on how the model should act (placed at the beginning)
        - `user` for messages from the user
        - `assistant` for messages from the model
    - `apply_chat_template` takes this list and returns a formatted sequence

In [ ]:
def generate_text_from_sample(model, processor, image, message, max_new_tokens=1024, device='cuda'):
    # Prepare the text input
    text_input = processor.apply_chat_template(
        message,  # Use the user message
        add_generation_prompt=True
    )

    # Display the text
    #print(text_input)

    # Prepare the image input
    image = Image.open(image).convert('RGB')
    image_inputs = []
    image_inputs.append([image]) # convert to list as processor requires it

    # Display the image
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Prepare the inputs for the model
    model_inputs = processor(
        text=text_input,
        images=image_inputs,
        return_tensors='pt', # Return PyTorch tensors
    ).to(device, dtype=torch.bfloat16)

    # Generate text with the model
    generated_token_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated token ids to remove the input token ids
    # Remove the original input tokens from the generated sequence
    trimmed_generated_token_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_token_ids)
    ]

    # Decode the output text
    # This converts the generated token IDs back into human-readable text
    output_text = processor.batch_decode(
        trimmed_generated_token_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0]

### 3.2. Test from test dataset

In [ ]:
image = test_dataset[1]['data'][1]['content'][0]['image']
message = [test_dataset[1]['data'][1]]
output = generate_text_from_sample(model, processor,image, message)
print(output)

### 3.3. Test on unseen image

In [ ]:
test_image = 'test-3.jpg'
message = [{'role': 'user','content': [
                {'type': 'image'},
                {'type': 'text', 'text': user_message}]},]
output = generate_text_from_sample(model, processor,test_image, message)
print(output)